<a href="https://colab.research.google.com/github/ribesstefano/TACK/blob/main/notebooks/fast_TACK_models_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting PROTAC degradation with the TACK fast surrogates

The TACK *fast surrogates* score a PROTAC in a specific experimental context — target
protein, E3 ligase, cell line, assay and treatment time — and return three predictions
with calibrated uncertainty:

| Task | Output | Meaning |
| --- | --- | --- |
| `dmax` | `mean` ∈ [0, 1] | maximum degradation, as a fraction |
| `pdc50` | `mean` | potency, −log₁₀(DC50 / M) — 9.0 means DC50 = 1 nM |
| `activity` | `proba` ∈ [0, 1] | P(active): Dmax > 80% **and** DC50 < 1 µM |

Two objects do all the work:

* **`FusionData`** turns records (SMILES + context) into rows of the design matrix. Molecules
  are featurised on the fly, so you can score compounds that did not exist at training time;
  context blocks come from published embedding tables.
* **`XGBStackedEnsemble`** is a stacked gradient-boosted ensemble — one per task — that maps
  those rows to predictions. No `torch`, no GPU, milliseconds per molecule.

**In this notebook**

1. [Setup](#setup) — install the package and sign in to the Hugging Face Hub
2. [Load the models](#load) — one context encoder, three task ensembles
3. [Inspect what a record accepts](#inspect) — fields, cached vocabularies, tasks
4. [Predict held-out literature measurements](#predict) — nine PROTACs from four papers
5. [Read the uncertainty](#uncertainty) — what each output column means
6. [Screen a congeneric series](#screen) — one context, many molecules
7. [Where to go next](#next)

> **Runtime:** CPU only, a couple of minutes end to end. The first call downloads ~100 MB of
> embedding tables and model files.

<a name="setup"></a>
## 1. Setup

On Colab this installs TACK from the `fast-surrogates` branch; locally it assumes you already
ran `uv pip install -e ".[dev]"` and does nothing.

`OMP_NUM_THREADS=1` is set **before** any import: on macOS `torch` and `xgboost` ship their own
`libomp` and segfault when both run multi-threaded in one process.

In [1]:
import importlib.util
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")  # must precede the first xgboost/torch import

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    !pip install -q "git+https://github.com/ribesstefano/TACK.git"

<a name="load"></a>
## 2. Load the models

`FusionData.from_pretrained` downloads the published context embedding tables — cell line, POI,
E3 ligase and assay — into the local cache (`TACKAI_CACHE`, or `~/.cache/tackai`) and verifies
them against a manifest. One ensemble per task lives in its own subfolder of the same repo.

In [ ]:
from tackai.fusion import FusionData, XGBStackedEnsemble

TASKS = ("dmax", "pdc50", "activity")

context = FusionData.from_pretrained("ailab-bio/TACK-Context-Encoding")
ensembles = {
    task: XGBStackedEnsemble.from_pretrained("ailab-bio/TACK-Fast-Ensemble", subfolder=task)
    for task in TASKS
}

print(f"design matrix: {context.n_columns} columns")
for block, cols in context.blocks_indexes.items():
    print(f"  {block:<12} {len(cols):>5} columns")
print()
for task, ens in ensembles.items():
    print(f"{task:<9} {len(ens.models):>2} members, {ens.help()['task']}")

Fetching 27 files: 100%|██████████| 27/27 [00:03<00:00,  7.09it/s]


design matrix: 1400 columns
  fingerprint   1024 columns
  descriptors    216 columns
  e3              52 columns
  cell            47 columns
  poi             52 columns
  assay            8 columns
  assay_time       1 columns

dmax      25 members, regression
pdc50     25 members, regression
activity  25 members, classification


<a name="inspect"></a>
## 3. Inspect what a record accepts

Both classes carry a `help()` method, so you never have to read the source to build a record or
to find out what `predict` returns.

In [4]:
def show(obj, indent=0):
    """Print a nested help() dict as an indented tree."""
    pad = "  " * indent
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, dict):
                print(f"{pad}{key}:")
                show(value, indent + 1)
            else:
                print(f"{pad}{key}: {value}")
    else:
        print(f"{pad}{obj}")


show(context.help())

record_fields: ['smiles', 'poi_seq', 'e3_seq', 'cell_id', 'assay', 'assay_time']
tasks:
  dmax:
    type: regression
    label: Dmax (fraction)
    support: (0.0, 1.0)
  pdc50:
    type: regression
    label: pDC50
    support: None
  activity:
    type: binary
    label: P(active)
    support: (0.0, 1.0)


`poi_seq`, `e3_seq`, `cell_id` and `assay` are looked up in the cached tables, so each has a
fixed vocabulary you can inspect. The behaviour on a miss is deliberately asymmetric:

* **cell line** — blank gets the *unknown* vector; an accession that is simply absent raises,
  because silently substituting *unknown* for a real cell line would hide the mistake.
* **assay** — free text is canonised, and an unseen type falls back to *unknown* with a warning.
* **POI / E3** — an unseen sequence raises; supply its 640-d embedding with
  `context.encoder.register_sequence(...)` if you need one.

In [5]:
for block in ("cell_id", "poi_seq", "e3_seq"):
    print(f"{block:<9} {context.help(block=block)['n_keys']:>6} cached keys")

print()
show(context.help(block="assay", return_keys=True))

cell_id      201 cached keys
poi_seq      215 cached keys
e3_seq       215 cached keys

block: assay
n_keys: 20
keys: ['capillary immunoassay', 'dot blot', 'egfp/mcherry reporter assay', 'elisa', 'elisa + western blot', 'flow cytometry', 'fluorescence imaging', 'fluorescent reporter', 'fluoresyn nanobody biosensor (gfp immunostaining)', 'hibit', 'high-content imaging', 'htrf', 'in-cell immunoassay', 'in-cell immunoassay + western blot', 'incell hunter', 'msd', 'nanoluc reporter', 'unknown', 'viability assay', 'western blot']


<a name="predict"></a>
## 4. Predict held-out literature measurements

Nine PROTACs from four papers, none of them in the training data, spanning five targets, both
common E3 ligases, two assay types and four orders of magnitude of potency — including two
deliberately awkward cases: compound **23** degrades only 37% (low Dmax despite a decent DC50)
and compound **7** has a DC50 of 10 µM (inactive by potency alone).

| Compound | Target | E3 | Cell line | Assay | DC50 | Dmax | Source |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 11 | BRD4 | CRBN | HeLa | Western blot | 0.63 nM | 97% | [10.1021/acs.jmedchem.6b01912](https://doi.org/10.1021/acs.jmedchem.6b01912) |
| NC-1 | BTK | CRBN | MINO | Western blot | 2.2 nM | 97% | [10.1021/jacs.9b13907](https://doi.org/10.1021/jacs.9b13907) |
| ACBI2 | SMARCA2 | VHL | RKO | Capillary immunoassay | 1.0 nM | 81% | [10.1038/s41467-022-33430-6](https://doi.org/10.1038/s41467-022-33430-6) |
| MS67 | WDR5 | VHL | MV4-11 | Western blot | 3.7 nM | 94% | [10.1126/scitranslmed.abj1578](https://doi.org/10.1126/scitranslmed.abj1578) |
| 9 | XIAP | VHL | MM1.S | Western blot | 0.70 nM | 99% | [10.1021/acs.jmedchem.2c01817](https://doi.org/10.1021/acs.jmedchem.2c01817) |
| 10 | HDAC4 | VHL | Jurkat E6.1 | MSD | 4.0 nM | 85% | [10.1021/acs.jmedchem.2c01149](https://doi.org/10.1021/acs.jmedchem.2c01149) |
| 23 | SMARCA2 | VHL | RKO | Capillary immunoassay | 39 nM | 37% | [10.1038/s41467-022-33430-6](https://doi.org/10.1038/s41467-022-33430-6) |
| 5 | SMARCA2 | VHL | RKO | Capillary immunoassay | 78 nM | 46% | [10.1038/s41467-022-33430-6](https://doi.org/10.1038/s41467-022-33430-6) |
| 7 | SMARCA2 | VHL | RKO | Capillary immunoassay | 10423 nM | 93% | [10.1038/s41467-022-33430-6](https://doi.org/10.1038/s41467-022-33430-6) |

In [6]:
import pandas as pd

pd.set_option("display.width", 200, "display.max_columns", 50)

# compound, target gene + UniProt, E3 + UniProt, cell line + Cellosaurus accession, assay,
# treatment time (h), measured DC50 (nM), measured Dmax (%), SMILES
MEASUREMENTS = [
    ("11",    "BRD4",    "O60885", "CRBN", "Q96SW2", "HeLa",        "CVCL_0030", "Western blot",          24,     0.63, 97, "Cc1sc2c(c1C)C(c1ccc(Cl)cc1)=N[C@@H](CC(=O)Nc1ccc(OCCOCCOCCOCCNc3cccc4c3C(=O)N(C3CCC(=O)NC3=O)C4=O)cc1)c1nnc(C)n1-2"),
    ("NC-1",  "BTK",     "Q06187", "CRBN", "Q96SW2", "MINO",        "CVCL_UW35", "Western blot",          24,     2.20, 97, "Nc1ncnc2c1c(-c1ccc(Oc3ccccc3)cc1)nn2[C@H]1CCCN(C(=O)CCCCCCOCCOCCOCCOCCOc2cccc3c2C(=O)N(C2CCC(=O)NC2=O)C3=O)C1"),
    ("ACBI2", "SMARCA2", "P51531", "VHL",  "P40337", "RKO",         "CVCL_0504", "Capillary immunoassay", 18,     1.00, 81, "Cc1ncsc1-c1ccc([C@H](CC[C@H](C)CCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)cc1"),
    ("MS67",  "WDR5",    "P61964", "VHL",  "P40337", "MV4-11",      "CVCL_0064", "Western blot",          18,     3.70, 94, "Cc1ncsc1-c1ccc([C@H](C)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)CNC(=O)c2ccc(F)c(-c3ccc(N4C[C@@H](C)N(C)[C@@H](C)C4)c(NC(=O)c4c[nH]c(=O)cc4C(F)(F)F)c3)c2)C(C)(C)C)cc1"),
    ("9",     "XIAP",    "P98170", "VHL",  "P40337", "MM1.S",       "CVCL_8792", "Western blot",          16,     0.70, 99, "CN[C@@H](C)C(=O)N[C@H](C(=O)N1C[C@@H](Oc2cccc(OCCCCC(=O)N[C@H](C(=O)N3C[C@H](O)C[C@H]3C(=O)N[C@@H](C)c3ccc(-c4scnc4C)cc3)C(C)(C)C)c2)C[C@H]1C(=O)N[C@@H]1CCCc2ccccc21)C1CCCCC1"),
    ("10",    "HDAC4",   "P56524", "VHL",  "P40337", "Jurkat E6.1", "CVCL_0367", "MSD",                   24,     4.00, 85, "CCCN(CCCCOCCOCCOCCOCC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1C(=O)NCc1ccc(-c2scnc2C)cc1)C(C)(C)C)C[C@@H](C)NC(=O)c1ccc(-c2noc(C(F)(F)F)n2)cc1"),
    ("23",    "SMARCA2", "P51531", "VHL",  "P40337", "RKO",         "CVCL_0504", "Capillary immunoassay", 18,    39.00, 37, "Cc1ncsc1-c1ccc(CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)c(OCCCCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)c1"),
    ("5",     "SMARCA2", "P51531", "VHL",  "P40337", "RKO",         "CVCL_0504", "Capillary immunoassay", 18,    78.00, 46, "Cc1ncsc1-c1ccc(CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)c(OCCCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)c1"),
    ("7",     "SMARCA2", "P51531", "VHL",  "P40337", "RKO",         "CVCL_0504", "Capillary immunoassay", 18, 10423.00, 93, "Cc1ncsc1-c1ccc([C@H](CCCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)NC(=O)[C@@H]2C[C@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)cc1"),
]
COLUMNS = ["compound", "poi", "poi_acc", "e3", "e3_acc", "cell", "cell_id", "assay",
           "assay_time", "dc50_nM", "dmax_pct", "smiles"]

benchmark = pd.DataFrame(MEASUREMENTS, columns=COLUMNS)
benchmark[["compound", "poi", "e3", "cell", "assay", "assay_time", "dc50_nM", "dmax_pct"]]

,compound,poi,e3,cell,assay,assay_time,dc50_nM,dmax_pct
0,11,BRD4,CRBN,HeLa,Western blot,24,0.63,97
1,NC-1,BTK,CRBN,MINO,Western blot,24,2.20,97
2,ACBI2,SMARCA2,VHL,RKO,Capillary immunoassay,18,1.00,81
3,MS67,WDR5,VHL,MV4-11,Western blot,18,3.70,94
4,9,XIAP,VHL,MM1.S,Western blot,16,0.70,99
5,10,HDAC4,VHL,Jurkat E6.1,MSD,24,4.00,85
6,23,SMARCA2,VHL,RKO,Capillary immunoassay,18,39.00,37
7,5,SMARCA2,VHL,RKO,Capillary immunoassay,18,78.00,46
8,7,SMARCA2,VHL,RKO,Capillary immunoassay,18,10423.00,93


### Sequences from UniProt

The POI and E3 blocks are keyed by the **full amino-acid sequence**, not by accession, so fetch
the canonical sequence once per accession — it matches the cached tables exactly.

In [15]:
import functools
import urllib.request


@functools.lru_cache(maxsize=None)
def uniprot_sequence(accession: str) -> str:
    """Canonical amino-acid sequence for a UniProt accession."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    with urllib.request.urlopen(url) as response:
        fasta = response.read().decode()
    return "".join(fasta.splitlines()[1:])


records = [
    {
        "smiles": row.smiles,
        "poi_seq": uniprot_sequence(row.poi_acc),   # target protein
        "e3_seq": uniprot_sequence(row.e3_acc),     # E3 ligase
        "cell_id": row.cell_id,                     # Cellosaurus accession, "" if unknown
        "assay": row.assay,                         # free text, canonised internally
        "assay_time": float(row.assay_time),        # hours; omit for the 24 h default
    }
    for row in benchmark.itertuples()
]

X = context.transform(records)
print("design matrix:", X.shape)

design matrix: (9, 1400)


### Predict

One `predict` call per task over the same matrix. `make_targets` converts the raw literature
numbers into the same three targets the models were trained on, so measured and predicted sit
side by side.

In [8]:
from tackai.fusion.data import make_targets

measured = make_targets(benchmark.dmax_pct, benchmark.dc50_nM)
predicted = {task: ensembles[task].predict(X) for task in TASKS}

results = pd.DataFrame({
    "compound": benchmark.compound,
    "POI": benchmark.poi,
    "cell": benchmark.cell,
    "Dmax": measured["dmax"],
    "Dmax_pred": predicted["dmax"]["mean"],
    "Dmax_std": predicted["dmax"]["std"],
    "pDC50": measured["pdc50"],
    "pDC50_pred": predicted["pdc50"]["mean"],
    "pDC50_std": predicted["pdc50"]["std"],
    "active": measured["activity"],
    "P(active)": predicted["activity"]["proba"],
})
results.round(2)

,compound,POI,cell,Dmax,Dmax_pred,Dmax_std,pDC50,pDC50_pred,pDC50_std,active,P(active)
0,11,BRD4,HeLa,0.97,0.94,0.11,9.20,8.88,0.85,1.0,0.87
1,NC-1,BTK,MINO,0.97,0.87,0.11,8.66,8.01,0.87,1.0,0.83
2,ACBI2,SMARCA2,RKO,0.81,0.80,0.10,9.00,8.78,0.78,1.0,0.75
3,MS67,WDR5,MV4-11,0.94,0.87,0.13,8.43,8.31,0.74,1.0,0.78
4,9,XIAP,MM1.S,0.99,0.90,0.14,9.15,8.92,0.88,1.0,0.84
5,10,HDAC4,Jurkat E6.1,0.85,0.82,0.10,8.40,8.16,0.79,1.0,0.73
6,23,SMARCA2,RKO,0.37,0.70,0.11,7.41,7.39,0.80,0.0,0.38
7,5,SMARCA2,RKO,0.46,0.52,0.15,7.11,6.79,0.80,0.0,0.16
8,7,SMARCA2,RKO,0.93,0.82,0.10,4.98,8.81,0.76,0.0,0.75


<a name="uncertainty"></a>
## 5. Read the uncertainty

Every prediction comes with its own error bar, split into the part the models agree on (noise)
and the part they disagree about (which shrinks as the training data covers your context).
`help(show_extras=True)` spells out each key and the fitted hyper-parameters behind it.

In [9]:
for task in TASKS:
    print("=" * 88)
    print(task)
    print("=" * 88)
    show(ensembles[task].help(show_extras=True))
    print()

dmax
task: regression
output_keys: ['mean', 'std', 'std_noise', 'std_disagreement', 'lower', 'upper']
hyperparameters:
  lambda_: 10.0
  alpha: 0.1
  q_hat_: 1.1948499763953568
explanations:
  mean: Weighted mean of the base models' predictions (weights_, uniform until fit()).
  std: Total predictive uncertainty: sqrt(std_noise**2 + std_disagreement**2).
  std_noise: Aleatoric term: sqrt(weights_ @ sigmas_**2), a single value shared by every prediction (not per-row) because sigmas_ are per-model constants fit by fit(); zero (hence std_noise=0) before fit().
  std_disagreement: Epistemic term: the weighted spread of the base models' predictions around mean — shrinks as the ensemble agrees with itself, independent of sigmas_.
  lower: mean - q_hat_ * std, i.e. the lower end of a conformal interval targeting 1 - alpha = 90% coverage (alpha=0.1) on data exchangeable with the calibration set; q_hat_=1.1948499763953568 is that set's (1 - alpha)-quantile of |y - mean| / std.
  upper: mean + q

A wide `Dmax_std` or a `P(active)` near 0.5 is the model telling you the measurement is worth
making. Where an ensemble has been calibrated, regression tasks also return `lower`/`upper` —
a conformal interval at the ensemble's fitted coverage level.

In [10]:
flagged = results.assign(uncertain=lambda df: df["Dmax_std"] > df["Dmax_std"].median())
flagged[["compound", "POI", "Dmax", "Dmax_pred", "Dmax_std", "P(active)", "uncertain"]].round(2)

,compound,POI,Dmax,Dmax_pred,Dmax_std,P(active),uncertain
0,11,BRD4,0.97,0.94,0.11,0.87,False
1,NC-1,BTK,0.97,0.87,0.11,0.83,False
2,ACBI2,SMARCA2,0.81,0.80,0.10,0.75,False
3,MS67,WDR5,0.94,0.87,0.13,0.78,True
4,9,XIAP,0.99,0.90,0.14,0.84,True
5,10,HDAC4,0.85,0.82,0.10,0.73,False
6,23,SMARCA2,0.37,0.70,0.11,0.38,True
7,5,SMARCA2,0.46,0.52,0.15,0.16,True
8,7,SMARCA2,0.93,0.82,0.10,0.75,False


<a name="screen"></a>
## 6. Screen a congeneric series

For virtual screening the context is fixed and only the molecules change. Encode the context
**once** with `transform_context`, then `assemble` it with freshly featurised SMILES — the
per-molecule cost drops to fingerprint + descriptors.

The nine SMARCA2 degraders below come from the ACBI2 paper and share a scaffold; only the
linker differs. Measured DC50 spans 1 nM → 10 µM.

In [11]:
SERIES = [
    # compound, measured DC50 (nM), measured Dmax (%), SMILES
    ("ACBI2",     1.0, 81, "Cc1ncsc1-c1ccc([C@H](CC[C@H](C)CCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)cc1"),
    ("11",        1.0, 87, "Cc1ncsc1-c1ccc([C@H](CCCCCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)cc1"),
    ("10",        3.0, 82, "Cc1ncsc1-c1ccc([C@H](CO[C@H](C)CN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(N(C)C)CC2)C(C)(C)C)cc1"),
    ("9",         5.0, 76, "Cc1ncsc1-c1ccc([C@H](CO[C@H](C)CN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)cc1"),
    ("8",        10.0, 80, "Cc1ncsc1-c1ccc([C@H](COCCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)NC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)cc1"),
    ("23",       39.0, 37, "Cc1ncsc1-c1ccc(CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)c(OCCCCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)c1"),
    ("5",        78.0, 46, "Cc1ncsc1-c1ccc(CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)c(OCCCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)c1"),
    ("22",      148.0, 75, "Cc1ncsc1-c1ccc(CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)c(OCCOCCOCCOCCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)c1"),
    ("7",     10423.0, 93, "Cc1ncsc1-c1ccc([C@H](CCCN2CCC(c3ccc4c(c3)n(C3CCCC3)c3nc(=O)c5c(Br)cccc5n43)CC2)NC(=O)[C@@H]2C[C@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)cc1"),
]
series = pd.DataFrame(SERIES, columns=["compound", "dc50_nM", "dmax_pct", "smiles"])

# SMARCA2 degradation in RKO, capillary immunoassay, 18 h — encoded once.
assay_context = context.transform_context({
    "poi_seq": uniprot_sequence("P51531"),  # SMARCA2
    "e3_seq": uniprot_sequence("P40337"),   # VHL
    "cell_id": "CVCL_0504",                 # RKO
    "assay": "Capillary immunoassay",
    "assay_time": 18.0,
})

X_series = context.assemble(series.smiles, assay_context)
truth = make_targets(series.dmax_pct, series.dc50_nM)

ranked = pd.DataFrame({
    "compound": series.compound,
    "pDC50": truth["pdc50"],
    "pDC50_pred": ensembles["pdc50"].predict(X_series)["mean"],
    "Dmax": truth["dmax"],
    "Dmax_pred": ensembles["dmax"].predict(X_series)["mean"],
    "P(active)": ensembles["activity"].predict(X_series)["proba"],
}).sort_values("pDC50_pred", ascending=False)
ranked.round(2)

,compound,pDC50,pDC50_pred,Dmax,Dmax_pred,P(active)
1,11,9.00,8.93,0.87,0.84,0.73
8,7,4.98,8.81,0.93,0.82,0.75
0,ACBI2,9.00,8.78,0.81,0.80,0.75
2,10,8.52,8.43,0.82,0.81,0.80
3,9,8.30,8.31,0.76,0.80,0.23
4,8,8.00,7.92,0.80,0.80,0.23
5,23,7.41,7.39,0.37,0.70,0.38
7,22,6.83,7.02,0.75,0.72,0.19
6,5,7.11,6.79,0.46,0.52,0.16


What matters for screening is the *ordering*, not the absolute values — Spearman ρ over the
series says how well the predicted ranking recovers the measured one.

In [12]:
from scipy.stats import spearmanr

for task, column in (("pdc50", "pDC50"), ("dmax", "Dmax")):
    rho = spearmanr(ranked[column], ranked[f"{column}_pred"]).statistic
    print(f"{task:<6} Spearman rho = {rho:+.2f}")

pdc50  Spearman rho = +0.50
dmax   Spearman rho = +0.92


<a name="next"></a>
## 7. Where to go next

**Scoring your own compounds.** Build one record per (molecule, context) pair and call
`context.transform(records)`. The only hard requirement is that the POI and E3 sequences are in
the cached tables; if they are not, register the embedding yourself with
`context.encoder.register_sequence(block, sequence, embedding)`.

**Thresholds.** `activity` is defined by `DMAX_THR = 0.80` and `PDC50_THR = 6.0` in
`tackai.fusion.data` — they differ from the legacy thresholds in `tackai.config`. A single
measurement can establish *inactive*, but *active* needs both.

**Keep in mind.**

* Predictions are only as good as the context coverage: a cell line or assay the training set
  barely saw will show up as a wide `std`, which is the useful signal here.
* `assay_time` matters — the same PROTAC has a different Dmax at 4 h and at 24 h.
* The molecular features are computed on the fly and are bit-exact with the legacy pipeline's
  (Morgan radius 16, 1024 bits, 217 RDKit descriptors).

**Related material**

* [`notebooks/fusion_comparison.ipynb`](https://github.com/ribesstefano/TACK/blob/fast-surrogates/notebooks/fusion_comparison.ipynb) — the benchmark that selected these models
* [`tackai/fusion/`](https://github.com/ribesstefano/TACK/tree/fast-surrogates/tackai/fusion) — module docstrings carry the design rationale
* `tack predict` — the legacy ensemble CLI, for the original TACK checkpoints